# Processing of RO-Crate for publishing
This notebook will read and RO-Crate file from galaxy, extract the provenance information as PROV-O in JSON-LD and save it into a new version of the RO-Crate.

1. read galaxy ro-crate
2. read the workflow file to get the prospective provenance
3. read the workflow run to get the retrospective provenance
4. merge the provenances into a PROV-O graph
5. save the PROV-O graph in the RO-Crate
6. get a publishable package consisting of RO-Crate + context (a JSON-LD file). The context metadata as JSON-LD is intended to facilitate discovery and interoperability
7. publish package to Invenio RDM (Zenodo if possible)


In [6]:
import zipfile
from pathlib import Path
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD
import json

# library to support visualisation of graphs
from rdflib.extras.external_graph_libs import rdflib_to_networkx_graph

# pyvis used to display the graphs
from pyvis.network import Network


# Namespaces
PROV = Namespace("http://www.w3.org/ns/prov#")
SCHEMA = Namespace("http://schema.org/")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")

# read file from zipped file
def get_file_from_rocratezip(zip_file, file_name):
    #print(f"Load {file_name} from crate: {str(zip_file)}")
    roc_zip_file = zipfile.ZipFile(zip_file)
    return roc_zip_file.open(file_name)
    
# Read the zipped crate and get main metadata file
def load_rocrate(crate_file):
    #Load the RO-Crate metadata file.
    #print(f"Load Crate: {str(crate_file)}")
    roc_zip_file = zipfile.ZipFile(crate_file)
    f = get_file_from_rocratezip(crate_file, "ro-crate-metadata.json")
    crate = json.load(f)
    return crate["@graph"]

#Return all JSON-LD entities of a given @type.
def find_entities(graph, type_name):
    return [e for e in graph if type_name in e.get("@type", [])]

#Convert RO-Crate IDs to URIs.
def to_uri(crate_base, entity_id):
    return URIRef(crate_base + entity_id)

#Convert json-ld crate to turtle
def crate_to_turtle(crate_path):
    #print(f"Convert to turtle: {str(crate_path)}")
    g = Graph()
    roc_zip_file = zipfile.ZipFile(crate_path)
    f = roc_zip_file.open("ro-crate-metadata.json")
    g.parse(f,format="json-ld")
    out_ttl="ro-crate-metadata.ttl"
    g.serialize(destination=out_ttl, format="turtle")
    print(f"RO-Crate turtle: {str(out_ttl)}")

def get_workflow_json(ro_crate_zip):
    graph = load_rocrate(ro_crate_zip)
    roc_files = find_entities(graph, "File")
    workflow_file = ""
    for a in roc_files:
        if a['@id'][-3:] == ".ga" :
            #print (f"This is the Galaxy workflow file {a['@id']}")
            workflow_file = a['@id']
            break
    
    f = get_file_from_rocratezip(ro_crate_zip, workflow_file)
    
    wf_ga_json = json.load(f)
    return wf_ga_json

def get_invocation_json(ro_crate_zip):
    graph = load_rocrate(ro_crate_zip)
    roc_files = find_entities(graph, "File")
    invocation_file = ""
    for a in roc_files:
        if "invocation_attrs.txt" in a['@id']:
            #print(f"This is the invocation attributes file {a['@id']}")
            invocation_file = a['@id']
            break
    
    f = get_file_from_rocratezip(ro_crate_zip, invocation_file)
    
    invocation_json = json.load(f)
    return invocation_json

# extract steps from wf and invocation files to create prov plan
def extract_prov_json(crate_path,output_file="galaxy_run_prov.ttl"):
    workflow_file = ""
    invocation_file = "" 

    # Load workflow (prospective provenance)
    wf = get_workflow_json(crate_path)
    steps = wf.get("steps", {})

    # Load invocation (retrospective provenance)
    inv = get_invocation_json(crate_path)
    inv_steps = inv[0].get("steps", {})

    g = Graph()
    g.bind("prov", PROV)
    g.bind("schema", SCHEMA)
    g.bind("skos",SKOS)

    base = "/"

    # --- 1. Workflow definition as prov:Plan ---
    wf_uri = URIRef(base + "workflow")
    g.add((wf_uri, RDF.type, PROV.Plan))
    g.add((wf_uri, RDFS.label, Literal(wf.get("name", "Galaxy Workflow"))))

    # --- 2. Prospective provenance: workflow steps ---
    for step_id, step in steps.items():
        #print(step)
        step_uri = URIRef(base + f"step/{step_id}")
        g.add((step_uri, RDF.type, PROV.Activity))
        step_label = step.get("label") or step.get("tool_id")
        g.add((step_uri, RDFS.label, Literal(step_label)))

        # Link step to workflow plan
        g.add((wf_uri, PROV.hadPlan, step_uri))

        # Prospective inputs
        for inp in step.get("input_connections", {}).values():
            if isinstance(inp, dict) and "id" in inp:
                src = URIRef(base + f"step/{inp['id']}")
                g.add((step_uri, PROV.wasInformedBy, src))

        # Tool as SoftwareAgent (prospective)
        tool_id = step.get("tool_id")
        if tool_id:
            tool_uri = URIRef(base + f"tool/{tool_id}")
            tool_label = tool_id.split("/")[-2]
            tool_version = tool_id.split("/")[-1]
            g.add((tool_uri, RDF.type, PROV.SoftwareAgent))
            g.add((tool_uri, RDFS.label, Literal(tool_id)))
            g.add((step_uri, PROV.wasAssociatedWith, tool_uri))
            g.add((tool_uri, SKOS.prefLabel, Literal(tool_label)))
            g.add((tool_uri, SCHEMA.softwareVersion, Literal(tool_version)))
            
    # --- 3. Retrospective provenance: actual execution ---
    for inv_step in inv_steps:
        job = inv_step.get("job")
        
        if not job:
            continue
        step_index = f"{inv_step['order_index']}"
        job_id = job['encoded_id']
        
        act_uri = URIRef(base + f"run/{job_id}")
        g.add((act_uri, RDF.type, PROV.Activity))
        g.add((act_uri, RDFS.label, Literal(f"Execution of step {step_index}")))
    
        # Link execution to prospective step
        step_uri = URIRef(base + f"step/{step_index}")
        g.add((act_uri, PROV.wasAssociatedWith, step_uri))
    
        # Inputs
        for inp_name, dataset in inv_step.get("inputs", {}).items():
            ent_uri = URIRef(base + f"dataset/{dataset['id']}")
            g.add((ent_uri, RDF.type, PROV.Entity))
            g.add((act_uri, PROV.used, ent_uri))
    
        # Outputs
        step_outputs = inv_step.get("outputs")
        for an_output in step_outputs:
            out_name = an_output['output_name']
            dataset = an_output['dataset']
    
            ent_uri = URIRef(base + f"dataset/{dataset['encoded_id']}")
    
            g.add((ent_uri, RDF.type, PROV.Entity))
            g.add((ent_uri, PROV.wasGeneratedBy, act_uri))
            g.add((ent_uri, RDFS.label, Literal(out_name)))

    # --- Save ---
    g.serialize(destination=output_file, format="turtle")

    context = {
        "@vocab": "http://www.w3.org/ns/prov#",
        "prov": "http://www.w3.org/ns/prov#",
        "schema": "http://schema.org/",
        "skos": "http://www.w3.org/2004/02/skos/core#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "xsd": "http://www.w3.org/2001/XMLSchema#"
    }
    
    g.serialize(destination=output_file+".json", format="json-ld", 
                context=context, auto_compact=True, indent=2)
    
    print(f"PROV-O written to {output_file}")
    return g

# convert json crate to prov (first try)
def extract_prov(crate_path, output_file="galaxy_prov.ttl"):
    graph = load_rocrate(crate_path)

    # Base URI for generated PROV entities
    base = "/"

    g = Graph()
    g.bind("prov", PROV)
    g.bind("schema", SCHEMA)

    # --- 1. Extract datasets (prov:Entity) ---
    datasets = find_entities(graph, "File") + find_entities(graph, "Dataset")

    for d in datasets:
        uri = to_uri(base, d["@id"])
        g.add((uri, RDF.type, PROV.Entity))

        if "name" in d:
            g.add((uri, RDFS.label, Literal(d["name"])))

        if "encodingFormat" in d:
            g.add((uri, PROV.value, Literal(d["encodingFormat"])))

    # --- 2. Extract workflow actions (prov:Activity) ---
    actions = find_entities(graph, "CreateAction") + find_entities(graph, "Action")
    
    for a in actions:
        act_uri = to_uri(base, a["@id"])
        g.add((act_uri, RDF.type, PROV.Activity))

        # timestamps
        if "startTime" in a:
            g.add((act_uri, PROV.startedAtTime, Literal(a["startTime"], datatype=XSD.dateTime)))
        if "endTime" in a:
            g.add((act_uri, PROV.endedAtTime, Literal(a["endTime"], datatype=XSD.dateTime)))

        # inputs → prov:used
        for inp in a.get("object", []):
            inp_uri = to_uri(base, inp["@id"])
            g.add((act_uri, PROV.used, inp_uri))

        # outputs → prov:wasGeneratedBy
        for out in a.get("result", []):
            out_uri = to_uri(base, out["@id"])
            g.add((out_uri, PROV.wasGeneratedBy, act_uri))

    # --- 3. Extract software agents (Galaxy, tools) ---
    software = find_entities(graph, "SoftwareApplication")

    for s in software:
        agent_uri = to_uri(base, s["@id"])
        g.add((agent_uri, RDF.type, PROV.SoftwareAgent))

        if "name" in s:
            g.add((agent_uri, RDFS.label, Literal(s["name"])))

        # Link software to actions via prov:wasAssociatedWith
        for a in actions:
            if a.get("instrument", {}).get("@id") == s["@id"]:
                act_uri = to_uri(base, a["@id"])
                g.add((act_uri, PROV.wasAssociatedWith, agent_uri))

    # --- Save crate graph ---
    g.serialize(destination=output_file, format="turtle")
    
    print(f"PROV-O written to {output_file}")
        

input_folder = "./input"
file_name = "FeS2-Analysis.rocrate.zip"

file_path = Path(input_folder,file_name)



if file_path.exists():
    print(f"Working with {str(file_path)}")
    roc_zip_file = zipfile.ZipFile(file_path)
    
    ro_crate_contents = roc_zip_file.namelist()
    
    #print(f"The file contains {len(ro_crate_contents)} elements (files and folders)")
    #print(ro_crate_contents)
    f = roc_zip_file.open("ro-crate-metadata.json")
    content = f.read()    
    #for item in content:
    #    if ".ga" == content[item][:-3]:
    #        print(f"Workflow file: {item}")
    #f.close()
    #print(content)

crate_to_turtle(file_path)

extract_prov(file_path)

roc_graph = extract_prov_json(file_path)

Working with input\FeS2-Analysis.rocrate.zip
RO-Crate turtle: ro-crate-metadata.ttl
PROV-O written to galaxy_prov.ttl
PROV-O written to galaxy_run_prov.ttl


In [2]:
nx_graph = rdflib_to_networkx_graph(roc_graph) #load graph
print(f"Newtworkx {nx_graph} loaded successfully")

nt = Network("800px", "100%", notebook="True")
nt.from_nx(nx_graph)
nt.show_buttons(filter_=["physics"])
nt.show("nx.html")

Newtworkx Graph with 55 nodes and 88 edges loaded successfully
nx.html


In [3]:
from rdflib import Graph, Namespace, RDF
from graphviz import Digraph

PROV = Namespace("http://www.w3.org/ns/prov#")

dot = Digraph("prov")

for s, p, o in roc_graph:
    # classify nodes
    if (s, RDF.type, PROV.Entity) in roc_graph:
        dot.node(str(s), shape="box", style="filled", fillcolor="#fff2b2")
    elif (s, RDF.type, PROV.Activity) in roc_graph:
        dot.node(str(s), shape="ellipse", style="filled", fillcolor="#b2d8ff")
    elif (s, RDF.type, PROV.Agent) in roc_graph:
        dot.node(str(s), shape="hexagon", style="filled", fillcolor="#c2f0c2")

    # add edges with labels
    if p.startswith(PROV):
        dot.edge(str(s), str(o), label=p.split("#")[-1])

dot.render("prov_graph", format="png")

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
step
print(step.get("label", step["tool_id"]))

In [ ]:
step['tool_id']

In [ ]:
x = step.get("label", step["tool_id"])
x

In [ ]:
x

In [ ]:
step.keys()

In [ ]:
tool_id = 'toolshed.g2.bx.psu.edu/repos/muon-spectroscopy-computational-project/larch_artemis/larch_artemis/0.9.75+galaxy0'
tool_label = tool_id.split("/")[-2]
tool_version = tool_id.split("/")[-1]

In [ ]:
tool_label


In [ ]:
tool_version

In [ ]:
pip install graphviz